In [11]:
import pandas as pd
import numpy as np
import glob, os
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, classification_report, confusion_matrix

In [12]:
globbed = glob.glob(os.path.join('TrafficLabelling', '*.csv'))
# encoding = 'latin1' from Gemini, otherwise encountering error with decoding CSV data
dfs = [pd.read_csv(file, encoding='latin1', low_memory=False) for file in globbed]
data = pd.concat(dfs, ignore_index=True)

In [13]:
cleaned_columns = []
for col in data.columns:
    cleaned_columns.append(col.strip().lower().replace(' ', '_'))
data.columns = cleaned_columns

data['timestamp'] = pd.to_datetime(data['timestamp'], errors='coerce')
data['malicious'] = (data['label'] != 'BENIGN').astype(int)
data.drop(columns=['label'], inplace=True)
data.replace([np.inf, -np.inf], np.nan, inplace=True)

data.dropna(inplace=True)
data.sort_values('timestamp', inplace=True)


In [14]:
index = int(len(data) * 0.8)
train = data.iloc[:index].copy()
test = data.iloc[index:].copy()

to_remove = ['flow_id', 'source_ip', 'destination_ip', 'timestamp']
train.drop(to_remove, errors='ignore', inplace=True)
test.drop(to_remove, errors='ignore', inplace=True)

x_train = train.drop(columns=['malicious'])
y_train = train['malicious']

x_test = test.drop(columns=['malicious'])
y_test = test['malicious']

non_numeric_cols = x_train.select_dtypes(exclude=[np.number]).columns.tolist()
x_train.drop(columns=non_numeric_cols, inplace=True)
x_test.drop(columns=non_numeric_cols, inplace=True)

In [15]:
scaler = StandardScaler()
x_train_scaler = scaler.fit_transform(x_train)
x_test_scaler = scaler.transform(x_test)

logistic_regression = LogisticRegression(class_weight='balanced', max_iter=1000)
logistic_regression.fit(x_train_scaler, y_train)

prediction = logistic_regression.predict(x_test_scaler)
probability = logistic_regression.predict_proba(x_test_scaler)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test, probability)
fpr_max_index = np.where(fpr <= 0.01)[0][-1]
actual_recall, actual_fpr, actual_threshold = tpr[fpr_max_index], fpr[fpr_max_index], thresholds[fpr_max_index]

In [20]:
print("Logistic Regression Results at 1% FPR:")
print(f"  Actual FPR: {actual_fpr * 100}%")
print(f"  Recall: {actual_recall * 100}%")
print(f"  Threshold: {actual_threshold * 100}%")
print()
print("Classification Report Results at 50% FPR:")
print(classification_report(y_test, prediction))

Logistic Regression Results at 1% FPR:
  Actual FPR: 0.9996820799039176%
  Recall: 63.87038036147828%
  Threshold: 86.68793015565942%

Classification Report Results at 50% FPR:
              precision    recall  f1-score   support

           0       0.95      0.92      0.93    311399
           1       0.84      0.90      0.87    148280

    accuracy                           0.91    459679
   macro avg       0.90      0.91      0.90    459679
weighted avg       0.91      0.91      0.91    459679

